## Purpose of this notebook

This notebook supports the DBRepo-related part of the assignment by creating and verifying all required DBRepo views through the REST API.

The created views are:

1. `vw_accident_casualty_ml_features`
2. `vw_environmental_severity_features`
3. `vw_binary_severity_features`
4. `vw_severity_by_weather`
5. `vw_severity_by_lighting`
6. `vw_severity_by_road_surface`
7. `vw_severity_by_vehicle`

The notebook first loads DBRepo table and column metadata, then builds REST API view payloads using table IDs and column IDs. Existing views are not deleted or recreated. If a view already exists, it is reported as `already_exists`; if it is missing, it is created through the REST API.

The final verification table confirms that all seven required views exist in DBRepo and records their DBRepo view identifiers. This verification table should be used as proof for the assignment report.

## 1. Connect to DBRepo

This section defines the DBRepo test endpoint and the target database ID used in this assignment.

The notebook asks for the DBRepo username and password at runtime using `input()` and `getpass()`. This avoids hardcoding credentials in the notebook and prevents passwords from being committed to GitHub.

The credentials are then used to create an HTTP Basic Authentication object for all REST API requests.

In [1]:
from getpass import getpass
from pathlib import Path
from io import StringIO
import json
import requests
from requests.auth import HTTPBasicAuth
import pandas as pd

DBREPO_ENDPOINT = "https://test.dbrepo.tuwien.ac.at"
BASE_URL = f"{DBREPO_ENDPOINT}/api/v1"
database_id = "b23492bd-f66d-4663-a1f5-f296767dbbdc"

username = input("DBRepo username: ").strip()
password = getpass("DBRepo password: ")
auth = HTTPBasicAuth(username, password)

print("DBRepo endpoint:", DBREPO_ENDPOINT)
print("Database ID:", database_id)

DBRepo endpoint: https://test.dbrepo.tuwien.ac.at
Database ID: b23492bd-f66d-4663-a1f5-f296767dbbdc


## 2. REST helpers

In [2]:
def api_get(path, params=None):
    url = f"{BASE_URL}{path}"
    r = requests.get(
        url,
        auth=auth,
        params=params,
        headers={"Accept": "application/json"},
        timeout=60,
    )
    if not r.ok:
        print("GET failed:", url)
        print("Status:", r.status_code)
        print(r.text[:4000])
        r.raise_for_status()
    return r.json() if r.text.strip() else None


def api_post(path, payload):
    url = f"{BASE_URL}{path}"
    r = requests.post(
        url,
        json=payload,
        auth=auth,
        headers={"Accept": "application/json", "Content-Type": "application/json"},
        timeout=120,
    )
    if not r.ok:
        print("POST failed:", url)
        print("Status:", r.status_code)
        print("Payload sent:")
        display(payload)
        print("Response:")
        print(r.text[:4000])
        r.raise_for_status()
    return r.json() if r.text.strip() else None


def api_delete(path):
    url = f"{BASE_URL}{path}"
    r = requests.delete(url, auth=auth, headers={"Accept": "application/json"}, timeout=60)
    if not r.ok and r.status_code != 404:
        print("DELETE failed:", url)
        print("Status:", r.status_code)
        print(r.text[:4000])
        r.raise_for_status()
    return r.status_code


def obj_get(obj, key, default=None):
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)


def as_list(payload):
    """DBRepo may return a list directly or wrap it under content/items/data."""
    if isinstance(payload, list):
        return payload
    if isinstance(payload, dict):
        for key in ["content", "items", "data", "results", "tables", "views"]:
            if isinstance(payload.get(key), list):
                return payload[key]
    return []

## 3. Fetch tables and columns

The DBRepo REST API does not create views from plain SQL text in this workflow. Instead, the view payload must reference DBRepo table IDs and column IDs.

This section therefore loads the available tables from the selected DBRepo database and builds a mapping between table names and their DBRepo UUIDs.

The notebook also fetches the column metadata for each required table. This allows the later view payloads to use the correct DBRepo column IDs instead of hardcoded values.

The required base and lookup tables are:

* `accident_casualty`
* `accident`
* `person`
* `vehicle`
* `road_class`
* `road_surface`
* `lighting`
* `weather`
* `sex`
* `casualty_class`
* `casualty_severity`

If any required table is missing, the notebook prints a warning so the issue can be checked before creating the views.

In [3]:
tables_payload = api_get(f"/database/{database_id}/table")
tables = as_list(tables_payload)

tables_df = pd.DataFrame([
    {"name": obj_get(t, "name"), "id": obj_get(t, "id")}
    for t in tables
]).sort_values("name")

display(tables_df)

table_name_to_id = dict(zip(tables_df["name"], tables_df["id"]))

required_tables = [
    "accident_casualty",
    "accident",
    "person",
    "vehicle",
    "road_class",
    "road_surface",
    "lighting",
    "weather",
    "sex",
    "casualty_class",
    "casualty_severity",
]

available_required_tables = [t for t in required_tables if t in table_name_to_id]
missing_tables = [t for t in required_tables if t not in table_name_to_id]

print("Available required tables:", available_required_tables)
if missing_tables:
    print("WARNING: Some lookup tables are missing. The notebook will still create simplified views where possible:")
    for t in missing_tables:
        print("-", t)
else:
    print("All required base/lookup tables are present.")

,name,id
1,accident,fccb4bde-6dcb-4e63-9358-945d7b1c93fc
0,accident_casualty,9365361e-d7f8-42d3-9c8b-543cc8477e99
5,casualty_class,f6df0087-e822-43e0-b784-86a05bd61db6
4,casualty_severity,5f8d9948-b3de-4c9a-8cce-55adb93750fb
9,lighting,66486c7c-6659-4083-9489-c018ed0d8900
2,person,872ede6c-f23d-4129-a179-3094b4a596fe
6,road_class,720ad88b-6d0a-4208-9349-fc30f4640896
7,road_surface,627bd0c6-c7d2-44a8-93ce-6e9004762eb0
3,sex,af0c82b5-29d1-4376-b4dc-587a7daf8d6b
8,vehicle,0cae0220-4b17-414e-b163-8113cc89b900


Available required tables: ['accident_casualty', 'accident', 'person', 'vehicle', 'road_class', 'road_surface', 'lighting', 'weather', 'sex', 'casualty_class', 'casualty_severity']
All required base/lookup tables are present.


In [4]:
table_detail_cache = {}
column_id_cache = {}


def get_table_detail(table_name):
    if table_name not in table_name_to_id:
        raise KeyError(f"Table not found in DBRepo: {table_name}")
    if table_name not in table_detail_cache:
        table_id = table_name_to_id[table_name]
        table_detail_cache[table_name] = api_get(f"/database/{database_id}/table/{table_id}")
    return table_detail_cache[table_name]


def get_columns(table_detail):
    columns = obj_get(table_detail, "columns", [])
    if not columns and isinstance(table_detail, dict):
        columns = table_detail.get("schema", {}).get("columns", [])
    return columns or []


def column_id(table_name, column_name):
    key = (table_name, column_name)
    if key in column_id_cache:
        return column_id_cache[key]

    detail = get_table_detail(table_name)
    columns = get_columns(detail)
    available = []
    for c in columns:
        cname = obj_get(c, "name")
        cid = obj_get(c, "id")
        available.append(cname)
        if cname == column_name:
            column_id_cache[key] = cid
            return cid

    raise KeyError(
        f"Column not found: {table_name}.{column_name}\n"
        f"Available columns: {available}"
    )


# Display available columns for debugging.
for table_name in available_required_tables:
    detail = get_table_detail(table_name)
    rows = [
        {"table": table_name, "column": obj_get(c, "name"), "id": obj_get(c, "id")}
        for c in get_columns(detail)
    ]
    print("\n", table_name)
    display(pd.DataFrame(rows))


 accident_casualty


,table,column,id
0,accident_casualty,reference_number,7a62a2e0-72a0-474c-bab3-c301d94fb5f0
1,accident_casualty,person_id,bf8af9af-df7e-4af9-ae2b-8641cc849f9a
2,accident_casualty,casualty_class_id,4d272416-6267-4e0b-9a33-58d95c8eeb40
3,accident_casualty,casualty_severity_id,fa47d1a8-42b2-4270-ab20-5431711d4e42
4,accident_casualty,vehicle_id,232c31c2-6fe7-4085-b414-122cc9164f92



 accident


,table,column,id
0,accident,reference_number,cef1e897-0646-45ad-8275-c7b624ef95de
1,accident,easting,9e3bbaa1-3ab8-4850-beb9-3ba55b2305ee
2,accident,northing,85573253-6384-4e5b-b8aa-963b6dc75625
3,accident,number_of_vehicles,57977d2b-29fa-4a63-a8cb-fb605846fcad
4,accident,date,351e4210-b9f5-4eef-a7b1-3e4edaefca96
5,accident,time,4bc272b4-0add-459a-bd45-992d360d6483
6,accident,road_class_id,87b64d39-d11c-4a4a-8c28-800793527784
7,accident,road_surface_id,f0b21c39-1619-4d47-a918-ba634ca7c2c0
8,accident,lighting_id,924e0223-f0ee-4494-8758-2c46fa483347
9,accident,weather_id,aca6641a-9189-459a-8652-24c7d7d8ec8e



 person


...

,table,column,id
0,sex,sex_id,e2f08c59-3c57-40cf-989e-7c2ada061ca6
1,sex,description,1c0470af-ce1a-44da-a78c-c7f14f75b249



 casualty_class


,table,column,id
0,casualty_class,casualty_class_id,08263f3a-5ca7-497a-abf9-844512c2b1bb
1,casualty_class,description,1249edac-98e4-4adf-8ef0-6e377807a065



 casualty_severity


,table,column,id
0,casualty_severity,casualty_severity_id,94657454-74c7-4da7-ad55-5b80e94fe736
1,casualty_severity,description,8ddee955-2730-41ab-9c14-9094ce02ce65


## 4. Build DBRepo `CreateViewDto` payloads

DBRepo views are created through a structured `CreateViewDto` JSON payload. This means the notebook cannot simply send a normal SQL `CREATE VIEW` statement to the API.

Instead, each view payload must specify:

* the view name,
* the base table,
* the selected columns using DBRepo column IDs,
* the join definitions using DBRepo table IDs and column IDs,
* optional filters and ordering rules.

The helper functions in this section make the payload creation easier:

* `col()` creates a selected column entry from a table name and column name.
* `join()` creates a join condition between two tables.
* `make_view_payload()` combines the selected columns, joins, and base table into the final REST API payload.

This approach avoids hardcoding DBRepo UUIDs manually and reduces errors when building the view definitions.

In [5]:
def has_table(table_name):
    return table_name in table_name_to_id


def has_column(table_name, column_name):
    if not has_table(table_name):
        return False
    try:
        column_id(table_name, column_name)
        return True
    except Exception:
        return False


def col(table_name, column_name, alias=None):
    item = {"id": column_id(table_name, column_name)}
    if alias:
        item["alias"] = alias
    return item


def join(join_type, datasource_table, left_table, left_column, right_table, right_column):
    return {
        "type": join_type,
        "datasource_id": table_name_to_id[datasource_table],
        "conditionals": [
            {
                "column_id": column_id(left_table, left_column),
                "foreign_column_id": column_id(right_table, right_column),
            }
        ],
    }


def make_view_payload(view_name, base_table, columns, joins=None):
    return {
        "name": view_name,
        "query": {
            "columns": columns,
            "joins": joins or [],
            "filters": [],
            "orders": [],
            "datasource_ids": [table_name_to_id[base_table]],
        },
        "is_public": False,
        "is_schema_public": False,
    }

## 5. Define all seven API-compatible views

The first two are the same API view names already confirmed in DBRepo.

The remaining five use the same required names but avoid unsupported SQL-only operations:

- `vw_binary_severity_features`: row-level severity fields, without SQL `CASE`.
- `vw_severity_by_weather`: row-level weather/severity fields, without `COUNT(*)` and `GROUP BY`.
- `vw_severity_by_lighting`: row-level lighting/severity fields.
- `vw_severity_by_road_surface`: row-level road surface/severity fields.
- `vw_severity_by_vehicle_type`: row-level vehicle/severity fields.

You can still calculate counts later in Python from the exported views.

In [ ]:
view_payloads = {}

# View 1: API-compatible ML feature view.
view_payloads["vw_accident_casualty_ml_features"] = make_view_payload(
    view_name="vw_accident_casualty_ml_features",
    base_table="accident_casualty",
    columns=[
        col("accident_casualty", "reference_number"),
        col("accident_casualty", "person_id"),
        col("accident_casualty", "vehicle_id"),
        col("accident_casualty", "casualty_class_id"),
        col("accident_casualty", "casualty_severity_id"),
        col("accident", "easting"),
        col("accident", "northing"),
        col("accident", "number_of_vehicles"),
        col("accident", "date"),
        col("accident", "time"),
        col("accident", "road_class_id"),
        col("accident", "road_surface_id"),
        col("accident", "lighting_id"),
        col("accident", "weather_id"),
        col("person", "age"),
        col("person", "sex_id"),
    ],
    joins=[
        join("inner", "accident", "accident_casualty", "reference_number", "accident", "reference_number"),
        join("inner", "person", "accident_casualty", "person_id", "person", "person_id"),
    ],
)

# View 2: API-compatible environmental severity view.
view_payloads["vw_environmental_severity_features"] = make_view_payload(
    view_name="vw_environmental_severity_features",
    base_table="accident_casualty",
    columns=[
        col("accident_casualty", "reference_number"),
        col("accident_casualty", "casualty_severity_id"),
        col("accident", "date"),
        col("accident", "time"),
        col("accident", "number_of_vehicles"),
        col("accident", "road_class_id"),
        col("accident", "road_surface_id"),
        col("accident", "lighting_id"),
        col("accident", "weather_id"),
    ],
    joins=[
        join("inner", "accident", "accident_casualty", "reference_number", "accident", "reference_number"),
    ],
)

# View 3: API-compatible binary severity source view.
# API limitation: no CASE expression here. Compute binary_severity later in Python from casualty_severity_id/description.
view_payloads["vw_binary_severity_features"] = make_view_payload(
    view_name="vw_binary_severity_features",
    base_table="accident_casualty",
    columns=[
        col("accident_casualty", "reference_number"),
        col("accident_casualty", "person_id"),
        col("accident_casualty", "vehicle_id"),
        col("accident_casualty", "casualty_class_id"),
        col("accident_casualty", "casualty_severity_id"),
        col("accident", "number_of_vehicles"),
        col("accident", "date"),
        col("accident", "time"),
        col("accident", "road_class_id"),
        col("accident", "road_surface_id"),
        col("accident", "lighting_id"),
        col("accident", "weather_id"),
        col("person", "age"),
        col("person", "sex_id"),
    ],
    joins=[
        join("inner", "accident", "accident_casualty", "reference_number", "accident", "reference_number"),
        join("inner", "person", "accident_casualty", "person_id", "person", "person_id"),
    ],
)

# View 4: API-compatible severity-by-weather source view.
# API limitation: no COUNT/GROUP BY here. Aggregate later using pandas groupby.
view_payloads["vw_severity_by_weather"] = make_view_payload(
    view_name="vw_severity_by_weather",
    base_table="accident_casualty",
    columns=[
        col("accident_casualty", "reference_number"),
        col("accident_casualty", "casualty_severity_id"),
        col("accident", "weather_id"),
    ],
    joins=[
        join("inner", "accident", "accident_casualty", "reference_number", "accident", "reference_number"),
    ],
)

# View 5: API-compatible severity-by-lighting source view.
view_payloads["vw_severity_by_lighting"] = make_view_payload(
    view_name="vw_severity_by_lighting",
    base_table="accident_casualty",
    columns=[
        col("accident_casualty", "reference_number"),
        col("accident_casualty", "casualty_severity_id"),
        col("accident", "lighting_id"),
    ],
    joins=[
        join("inner", "accident", "accident_casualty", "reference_number", "accident", "reference_number"),
    ],
)

# View 6: API-compatible severity-by-road-surface source view.
view_payloads["vw_severity_by_road_surface"] = make_view_payload(
    view_name="vw_severity_by_road_surface",
    base_table="accident_casualty",
    columns=[
        col("accident_casualty", "reference_number"),
        col("accident_casualty", "casualty_severity_id"),
        col("accident", "road_surface_id"),
    ],
    joins=[
        join("inner", "accident", "accident_casualty", "reference_number", "accident", "reference_number"),
    ],
)

# View 7: API-compatible severity-by-vehicle-type source view.
# This can be created from accident_casualty alone because vehicle_id is already present there.
view_payloads["vw_severity_by_vehicle"] = make_view_payload(
    view_name="vw_severity_by_vehicle",
    base_table="accident_casualty",
    columns=[
        col("accident_casualty", "reference_number"),
        col("accident_casualty", "vehicle_id"),
        col("accident_casualty", "casualty_severity_id"),
    ],
    joins=[],
)

print("Prepared API payloads:", len(view_payloads))
print(list(view_payloads.keys()))

# Inspect one payload before sending.
example_name = "vw_severity_by_weather"
print("Example payload:", example_name)
display(view_payloads[example_name])

Prepared API payloads: 7
['vw_accident_casualty_ml_features', 'vw_environmental_severity_features', 'vw_binary_severity_features', 'vw_severity_by_weather', 'vw_severity_by_lighting', 'vw_severity_by_road_surface', 'vw_severity_by_vehicle_type']
Example payload: vw_severity_by_weather


{'name': 'vw_severity_by_weather',
 'query': {'columns': [{'id': '7a62a2e0-72a0-474c-bab3-c301d94fb5f0'},
   {'id': 'fa47d1a8-42b2-4270-ab20-5431711d4e42'},
   {'id': 'aca6641a-9189-459a-8652-24c7d7d8ec8e'}],
  'joins': [{'type': 'inner',
    'datasource_id': 'fccb4bde-6dcb-4e63-9358-945d7b1c93fc',
    'conditionals': [{'column_id': '7a62a2e0-72a0-474c-bab3-c301d94fb5f0',
      'foreign_column_id': 'cef1e897-0646-45ad-8275-c7b624ef95de'}]}],
  'filters': [],
  'orders': [],
  'datasource_ids': ['9365361e-d7f8-42d3-9c8b-543cc8477e99']},
 'is_public': False,
 'is_schema_public': False}

## 6. Create views via REST API

This section creates the DBRepo views using the REST API payloads defined above.

The variable `RECREATE_EXISTING_VIEWS` is set to `False` for safety. This means the notebook does not delete or overwrite existing views.

For each required view, the notebook first checks whether the view already exists in DBRepo:

* if the view already exists, it is reported as `already_exists`;
* if the view is missing, it is created using `POST /database/{database_id}/view`;
* if creation fails, the error is captured and shown in the results table.

This allows the notebook to be run multiple times safely without accidentally deleting existing DBRepo views.

In [ ]:
RECREATE_EXISTING_VIEWS = False


def get_existing_views():
    views_payload = api_get(f"/database/{database_id}/view")
    views = as_list(views_payload)
    return {obj_get(v, "name"): v for v in views}


def delete_view_by_name(view_name):
    existing = get_existing_views()
    view = existing.get(view_name)
    if not view:
        return False
    view_id = obj_get(view, "id")
    api_delete(f"/database/{database_id}/view/{view_id}")
    print("Deleted existing view:", view_name)
    return True

def create_view_rest(view_name, payload, recreate=False):
    if recreate:
        delete_view_by_name(view_name)

    existing = get_existing_views()
    if view_name in existing:
        print("Already exists:", view_name)
        return {
            "view": view_name,
            "status": "already_exists",
            "id": obj_get(existing[view_name], "id"),
            "detail": "View already exists in DBRepo",
        }

    try:
        created = api_post(f"/database/{database_id}/view", payload)
        print("Created via REST API:", view_name)
        return {
            "view": view_name,
            "status": "created",
            "id": obj_get(created, "id"),
            "detail": "Created with POST /database/{databaseId}/view",
        }
    except Exception as exc:
        print("Could not create via REST API:", view_name)
        print(type(exc).__name__, exc)
        return {
            "view": view_name,
            "status": "failed",
            "id": None,
            "detail": f"{type(exc).__name__}: {exc}",
        }


results = []
for view_name, payload in view_payloads.items():
    results.append(create_view_rest(view_name, payload, recreate=RECREATE_EXISTING_VIEWS))

results_df = pd.DataFrame(results)
display(results_df)

Already exists: vw_accident_casualty_ml_features
Already exists: vw_environmental_severity_features
Created via REST API: vw_binary_severity_features
Created via REST API: vw_severity_by_weather
Created via REST API: vw_severity_by_lighting
Created via REST API: vw_severity_by_road_surface
Created via REST API: vw_severity_by_vehicle_type


,view,status,id,detail
0,vw_accident_casualty_ml_features,already_exists,9bb88a33-e2e9-4736-8d3e-99f0ce023e47,View already exists in DBRepo
1,vw_environmental_severity_features,already_exists,229a6593-b0ef-4424-9663-0224eee1a4aa,View already exists in DBRepo
2,vw_binary_severity_features,created,3fa6fbd7-207d-4356-bf49-4089e6f30a04,Created with POST /database/{databaseId}/view
3,vw_severity_by_weather,created,d951054a-8c1f-44b9-90c2-ec965f9d7002,Created with POST /database/{databaseId}/view
4,vw_severity_by_lighting,created,5cc50c5d-65ca-4ad1-9526-e611bffa014d,Created with POST /database/{databaseId}/view
5,vw_severity_by_road_surface,created,f5429f6c-a392-42dd-8667-c7cd78695286,Created with POST /database/{databaseId}/view
6,vw_severity_by_vehicle_type,created,a57a45ce-6109-403f-893e-69b474e2717f,Created with POST /database/{databaseId}/view


## 7. Verify all seven views exist

This table is the main screenshot/proof for the assignment.

In [8]:
existing_views = get_existing_views()
verification_rows = []

for view_name in view_payloads:
    view = existing_views.get(view_name)
    verification_rows.append({
        "view": view_name,
        "exists_in_dbrepo": view is not None,
        "id": obj_get(view, "id") if view else None,
    })

verification_df = pd.DataFrame(verification_rows)
display(verification_df)

missing = verification_df.loc[~verification_df["exists_in_dbrepo"], "view"].tolist()
if missing:
    print("Missing views:")
    for v in missing:
        print("-", v)
else:
    print("All seven required views exist in DBRepo.")

,view,exists_in_dbrepo,id
0,vw_accident_casualty_ml_features,True,9bb88a33-e2e9-4736-8d3e-99f0ce023e47
1,vw_environmental_severity_features,True,229a6593-b0ef-4424-9663-0224eee1a4aa
2,vw_binary_severity_features,True,3fa6fbd7-207d-4356-bf49-4089e6f30a04
3,vw_severity_by_weather,True,d951054a-8c1f-44b9-90c2-ec965f9d7002
4,vw_severity_by_lighting,True,5cc50c5d-65ca-4ad1-9526-e611bffa014d
5,vw_severity_by_road_surface,True,f5429f6c-a392-42dd-8667-c7cd78695286
6,vw_severity_by_vehicle_type,True,a57a45ce-6109-403f-893e-69b474e2717f


All seven required views exist in DBRepo.


## 8. Optional preview

Creation is proven by the metadata verification table above. Preview can fail depending on DBRepo endpoint response format, so this cell does not affect the assignment result.

In [9]:
def preview_view_metadata_only(view_name):
    existing = get_existing_views()
    view = existing.get(view_name)
    if not view:
        print("View does not exist:", view_name)
        return None
    print("View exists:", view_name)
    print("View ID:", obj_get(view, "id"))
    return view

preview_view_metadata_only("vw_accident_casualty_ml_features")

View exists: vw_accident_casualty_ml_features
View ID: 9bb88a33-e2e9-4736-8d3e-99f0ce023e47


{'id': '9bb88a33-e2e9-4736-8d3e-99f0ce023e47',
 'name': 'vw_accident_casualty_ml_features',
 'query': 'select `road_traffic_accident_severity_prediction_leeds_tht8`.`accident`.`lighting_id`, `road_traffic_accident_severity_prediction_leeds_tht8`.`accident`.`road_surface_id`, `road_traffic_accident_severity_prediction_leeds_tht8`.`person`.`age`, `road_traffic_accident_severity_prediction_leeds_tht8`.`accident`.`date`, `road_traffic_accident_severity_prediction_leeds_tht8`.`accident_casualty`.`person_id`, `road_traffic_accident_severity_prediction_leeds_tht8`.`accident`.`time`, `road_traffic_accident_severity_prediction_leeds_tht8`.`accident`.`road_class_id`, `road_traffic_accident_severity_prediction_leeds_tht8`.`accident_casualty`.`reference_number`, `road_traffic_accident_severity_prediction_leeds_tht8`.`accident_casualty`.`casualty_severity_id`, `road_traffic_accident_severity_prediction_leeds_tht8`.`accident`.`easting`, `road_traffic_accident_severity_prediction_leeds_tht8`.`acciden